# Wesely (1989) gas dry deposition

This notebook runs the Wesely (1989) scheme from `quacs.drydep.wesely` and plots the results.
All physics is in `quacs/drydep/wesely/wesely.py`. This notebook only sets the inputs and makes plots.

Original notebook: Saeideh Mohammadi (University of Iowa).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from quacs.drydep.wesely import LAND_NAMES, run_box_model, wesely_gas

## Inputs

In [ ]:
species_list = ["O3", "SO2", "NO2", "HNO3", "H2O2", "CH2O", "NH3"]

MET = dict(
    sfc_temp=298.0,  # K
    air_temp=296.0,  # K
    pressure_sfc=101325.0,  # Pa
    pressure_10m=100000.0,  # Pa
    wind_speed=5.0,  # m/s
    spec_hum=0.01,  # kg/kg
    solar_flux=400.0,  # W/m2
    month=7,
    lat=40.0,  # degrees
)

# Land-use fractions, in the order of LAND_NAMES
frac_landuse = np.array([0.05, 0.30, 0.10, 0.25, 0.10, 0.05, 0.05, 0.0, 0.05, 0.05, 0.0])

out = run_box_model(species_list, MET, frac_landuse, box_height_m=1000.0)

## Concentration decay

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for sp in species_list:
    ax.plot(out["times"] / 3600, out["concs"][sp], lw=2, label=sp)
ax.set_xlabel("Time (hours)")
ax.set_ylabel("C/C0")
ax.set_title("Concentration decay")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Deposition velocity and resistances by species

In [ ]:
vd = [out["Vd"][sp] for sp in species_list]  # cm/s
ra = [out["results"][sp]["Ra"] for sp in species_list]  # s/m
rb = [out["results"][sp]["Rb"] for sp in species_list]
rc = [out["results"][sp]["Rc"] for sp in species_list]

# Rc can be very large for near-inert species. Cap it for display only.
RC_DISPLAY_CAP = 400
rc_disp = [min(r, RC_DISPLAY_CAP) for r in rc]

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(species_list, vd, height=0.6, color="C0")
for y, v in enumerate(vd):
    ax.text(v + 0.02, y, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlabel("V$_d$ (cm s$^{-1}$)")
ax.set_title("Deposition velocity by species")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()

# Series resistances add: Vd = 1/(Ra + Rb + Rc)
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(species_list))
ax.bar(x, ra, 0.6, label="R$_a$ (aerodynamic)", color="C0")
ax.bar(x, rb, 0.6, bottom=ra, label="R$_b$ (quasi-laminar)", color="C1")
ax.bar(
    x,
    rc_disp,
    0.6,
    bottom=np.add(ra, rb),
    label=f"R$_c$ (surface, capped {RC_DISPLAY_CAP})",
    color="C2",
)
ax.set_xticks(x)
ax.set_xticklabels(species_list)
ax.set_ylabel("Resistance (s m$^{-1}$)")
ax.set_title("Resistance decomposition by species")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## O3 deposition velocity by land type

In [ ]:
o3 = out["results"]["O3"]
vd_land = np.full(len(LAND_NAMES), np.nan)
for i in range(len(LAND_NAMES)):
    r_tot = o3["ra_lt"][i] + o3["rb_lt"][i] + o3["rc_lt"][i]
    if frac_landuse[i] > 0 and r_tot > 0:
        vd_land[i] = 100.0 / r_tot  # 1/R in m/s -> cm/s
present = [i for i in range(len(LAND_NAMES)) if frac_landuse[i] > 0]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(len(present)), vd_land[present], color="C0")
ax.axhline(
    out["Vd"]["O3"],
    color="k",
    ls="--",
    lw=1.5,
    label=f"grid mean = {out['Vd']['O3']:.3f} cm s$^{{-1}}$",
)
ax.set_xticks(range(len(present)))
ax.set_xticklabels([LAND_NAMES[i] for i in present], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("O$_3$ V$_d$ (cm s$^{-1}$)")
ax.set_title("O$_3$ V$_d$ by land type")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## Seasonal cycle of the grid-mean O3 deposition velocity

In [ ]:
season_month = {"Summer": 7, "Autumn": 10, "Late\nAutumn": 11, "Winter": 1, "Spring": 4}
met_no_month = {k: v for k, v in MET.items() if k != "month"}
vd_season = [
    wesely_gas("O3", frac_landuse=frac_landuse, month=m, **met_no_month)["Vd"]
    for m in season_month.values()
]

fig, ax = plt.subplots(figsize=(6, 4))
xs = np.arange(len(season_month))
ax.plot(xs, vd_season, "o-", color="C0", lw=2, ms=7)
ax.set_xticks(xs)
ax.set_xticklabels(list(season_month.keys()))
ax.set_ylabel("O$_3$ V$_d$ (cm s$^{-1}$)")
ax.set_title("Modelled seasonal cycle of grid-mean O$_3$ V$_d$")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()